In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
save_dir = "/content/drive/MyDrive/NLP_Project_Preprocessing"

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [ ]:
!pip install -q tf-keras

In [ ]:
import os

os.makedirs(save_dir, exist_ok=True)
print(os.listdir(save_dir))

['sentiment_encoder.pkl', 'stance_encoder.pkl', 'train_idx.npy', 'test_idx.npy', 'y_train_stance.npy', 'y_test_stance.npy', 'y_train_sentiment.npy', 'y_test_sentiment.npy', 'X_train.csv', 'X_test.csv', 'X_train_tokens.pkl', 'X_test_tokens.pkl', 'word_index.pkl', 'word2vec.model', 'X_train_pad.npy', 'X_test_pad.npy', 'embedding_matrix.npy', 'rnn_stance.keras', 'rnn_sentiment.keras', 'test_original_tweets.csv', 'train_original_tweets.csv', 'predictions.csv', 'bert_stance', 'lstm_stance.keras', 'config.json', 'tf_model.h5', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json']


In [ ]:
import pandas as pd

X_train = pd.read_csv(f"{save_dir}/X_train.csv")
X_test = pd.read_csv(f"{save_dir}/X_test.csv")

In [ ]:
print(X_train.shape)
print(X_train.head())
print(X_train.columns)

(1166475, 1)
                                  reconstructed_text
0       ukraine rejects russian neutrality proposals
1  citizens protesting zahedan injured clashes re...
2    pope francis send emissaries russia ukraine war
3  new spoke parents prosper adopted orphan broth...
4  live one month start war ukraine pan american ...
Index(['reconstructed_text'], dtype='object')


In [ ]:
print(len(X_train))
print(len(X_test))

1166475
291619


In [ ]:
lengths = X_train["reconstructed_text"].str.split().str.len()

print("Max:", lengths.max())
print("Mean:", lengths.mean())
print("95th percentile:", lengths.quantile(0.95))
print("99th percentile:", lengths.quantile(0.99))

Max: 88
Mean: 13.136587582245655
95th percentile: 25.0
99th percentile: 30.0


In [ ]:
!pip install transformers==4.49.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of t

In [ ]:
import numpy as np
import tensorflow as tf

from transformers import (
    AutoTokenizer,
    TFAutoModelForSequenceClassification
)

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

In [ ]:
y_train_sentiment = np.load(f"{save_dir}/y_train_sentiment.npy")
y_test_sentiment = np.load(f"{save_dir}/y_test_sentiment.npy")

In [ ]:
sample_size = 100000

X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train_sentiment,
    train_size=sample_size,
    stratify=y_train_sentiment,
    random_state=42
)

In [ ]:
X_train_text = X_train_sample["reconstructed_text"].tolist()
X_test_text = X_test["reconstructed_text"].tolist()

In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 32

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
train_encodings = tokenizer(
    X_train_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

test_encodings = tokenizer(
    X_test_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN
)

In [ ]:
train_encodings = {k: np.array(v) for k, v in train_encodings.items()}
test_encodings = {k: np.array(v) for k, v in test_encodings.items()}

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

all_train_encodings = train_encodings


train_idx, val_idx = train_test_split(
    np.arange(len(y_train_sample)),
    test_size=0.1,
    stratify=y_train_sample,
    random_state=42
)


In [ ]:
print("train_encodings:", len(train_encodings["input_ids"]))
print("all_train_encodings:", len(all_train_encodings["input_ids"]))
print("max train_idx:", train_idx.max())
print("max val_idx:", val_idx.max())

train_encodings: 100000
all_train_encodings: 100000
max train_idx: 99999
max val_idx: 99993


In [ ]:
train_encodings = {
    k: v[train_idx]
    for k, v in all_train_encodings.items()
}

val_encodings = {
    k: v[val_idx]
    for k, v in all_train_encodings.items()
}

y_train = y_train_sample[train_idx]
y_val = y_train_sample[val_idx]


In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    train_encodings,
    y_train
))

val_dataset = tf.data.Dataset.from_tensor_slices((
    val_encodings,
    y_val
))

test_dataset = tf.data.Dataset.from_tensor_slices((
    test_encodings,
    y_test_sentiment
))

In [ ]:
BATCH_SIZE = 16

train_dataset = (
    train_dataset
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
print(len(train_encodings["input_ids"]))
print(len(val_encodings["input_ids"]))
print(len(y_train_sample))
print(len(y_val))

90000
10000
100000
10000


In [ ]:
print(len(train_encodings["input_ids"]))
print(len(val_encodings["input_ids"]))
print(len(y_train))
print(len(y_val))

90000
10000
90000
10000


In [ ]:
print(len(y_train))
print(len(y_val))

print(tf.data.experimental.cardinality(train_dataset).numpy())
print(tf.data.experimental.cardinality(val_dataset).numpy())

90000
10000
5625
625


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_sample),
    y=y_train_sample
)

class_weights = {
    0: weights[0],
    1: weights[1],
    2: weights[2]
}

print(class_weights)

{0: np.float64(0.8395248289468161), 1: np.float64(0.5681495369581274), 2: np.float64(20.512820512820515)}


In [ ]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import tensorflow as tf
import transformers
import keras

print("TensorFlow:", tf.__version__)
print("Transformers:", transformers.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.20.0
Transformers: 4.49.0
Keras: 3.13.2


In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=2e-5
)

loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

In [ ]:
model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[early_stopping]
)

Epoch 1/10
5625/5625 [==============================] - 893s 150ms/step - loss: 0.4036 - accuracy: 0.8215 - val_loss: 0.3664 - val_accuracy: 0.8382
Epoch 2/10
5625/5625 [==============================] - 829s 147ms/step - loss: 0.2754 - accuracy: 0.8840 - val_loss: 0.3810 - val_accuracy: 0.8413
Epoch 3/10
5625/5625 [==============================] - 830s 147ms/step - loss: 0.1643 - accuracy: 0.9361 - val_loss: 0.5042 - val_accuracy: 0.8296
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


In [ ]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 999s 55ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.84      0.79      0.82    115785
     Neutral       0.85      0.89      0.87    171094
    Positive       0.54      0.57      0.56      4740

    accuracy                           0.84    291619
   macro avg       0.75      0.75      0.75    291619
weighted avg       0.84      0.84      0.84    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 91723  23999     63]
 [ 16918 151934   2242]
 [    39   1988   2713]]


##model 2


In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
5625/5625 [==============================] - 884s 148ms/step - loss: 0.5239 - accuracy: 0.7228 - val_loss: 0.5386 - val_accuracy: 0.7693
Epoch 2/10
5625/5625 [==============================] - 883s 157ms/step - loss: 0.3564 - accuracy: 0.8032 - val_loss: 0.4813 - val_accuracy: 0.7894
Epoch 3/10
5625/5625 [==============================] - 820s 146ms/step - loss: 0.2778 - accuracy: 0.8475 - val_loss: 0.4524 - val_accuracy: 0.7987
Epoch 4/10
5625/5625 [==============================] - 814s 145ms/step - loss: 0.2308 - accuracy: 0.8742 - val_loss: 0.4713 - val_accuracy: 0.8123
Epoch 5/10
5625/5625 [==============================] - 827s 147ms/step - loss: 0.1806 - accuracy: 0.9052 - val_loss: 0.5059 - val_accuracy: 0.8233
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 3.


In [ ]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 997s 54ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.77      0.88      0.82    115785
     Neutral       0.90      0.76      0.82    171094
    Positive       0.24      0.81      0.37      4740

    accuracy                           0.80    291619
   macro avg       0.64      0.82      0.67    291619
weighted avg       0.84      0.80      0.81    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[101355  13949    481]
 [ 29987 129472  11635]
 [   117    763   3860]]


##model 3

In [ ]:
class_weights = {
    0: 1.0,
    1: 0.7,
    2: 10.0
}

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
5625/5625 [==============================] - 833s 146ms/step - loss: 0.2385 - accuracy: 0.8854 - val_loss: 0.4495 - val_accuracy: 0.8244
Epoch 2/10
5625/5625 [==============================] - 815s 145ms/step - loss: 0.1805 - accuracy: 0.9184 - val_loss: 0.4794 - val_accuracy: 0.8250
Epoch 3/10
5625/5625 [==============================] - 812s 144ms/step - loss: 0.1326 - accuracy: 0.9432 - val_loss: 0.5203 - val_accuracy: 0.8297
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


In [ ]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 1014s 56ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.78      0.87      0.82    115785
     Neutral       0.89      0.81      0.85    171094
    Positive       0.43      0.61      0.51      4740

    accuracy                           0.83    291619
   macro avg       0.70      0.76      0.73    291619
weighted avg       0.84      0.83      0.83    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[100440  15246     99]
 [ 28433 138977   3684]
 [    81   1763   2896]]


##model 4

In [ ]:
class_weights = {
    0: 0.9,
    1: 0.7,
    2: 14.0
}

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
5625/5625 [==============================] - 952s 159ms/step - loss: 0.5093 - accuracy: 0.7593 - val_loss: 0.4080 - val_accuracy: 0.8202
Epoch 2/10
5625/5625 [==============================] - 872s 155ms/step - loss: 0.3475 - accuracy: 0.8293 - val_loss: 0.4550 - val_accuracy: 0.8045
Epoch 3/10
5625/5625 [==============================] - 868s 154ms/step - loss: 0.2692 - accuracy: 0.8697 - val_loss: 0.4290 - val_accuracy: 0.8243
Epoch 3: early stopping
Restoring model weights from the end of the best epoch: 1.


In [ ]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 1065s 58ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.79      0.85      0.82    115785
     Neutral       0.88      0.81      0.84    171094
    Positive       0.33      0.75      0.46      4740

    accuracy                           0.82    291619
   macro avg       0.67      0.80      0.71    291619
weighted avg       0.84      0.82      0.83    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 97901  17261    623]
 [ 26127 138249   6718]
 [    81   1096   3563]]


##train for entire data


In [ ]:
from sklearn.model_selection import train_test_split
import tensorflow as tf

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train_sentiment,
    test_size=0.1,
    stratify=y_train_sentiment,
    random_state=42
)

In [ ]:
X_train_text = X_train_final["reconstructed_text"].tolist()
X_val_text = X_val["reconstructed_text"].tolist()
X_test_text = X_test["reconstructed_text"].tolist()

In [ ]:
MAX_LEN = 32

train_encodings = tokenizer(
    X_train_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="tf"
)

val_encodings = tokenizer(
    X_val_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="tf"
)

test_encodings = tokenizer(
    X_test_text,
    truncation=True,
    padding="max_length",
    max_length=MAX_LEN,
    return_tensors="tf"
)

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    y_train_final
))

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    y_val
))

test_dataset = tf.data.Dataset.from_tensor_slices((
    dict(test_encodings),
    y_test_sentiment
))

In [ ]:
BATCH_SIZE = 16

train_dataset = (
    train_dataset
    .shuffle(10000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_dataset = (
    val_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    test_dataset
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
import os
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    CSVLogger
)

SAVE_DIR_1 = "/content/drive/MyDrive/New_BERT_Sentiment_Final"
os.makedirs(SAVE_DIR_1, exist_ok=True)


epoch_checkpoint = ModelCheckpoint(
    filepath=os.path.join(
        SAVE_DIR_1,
        "checkpoint_epoch_{epoch:02d}"
    ),
    monitor="val_loss",
    save_best_only=False,
    save_weights_only=False,
    save_freq="epoch",
    verbose=1
)

best_checkpoint = ModelCheckpoint(
    filepath=os.path.join(
        SAVE_DIR_1,
        "best_model"
    ),
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

csv_logger = CSVLogger(
    os.path.join(SAVE_DIR_1, "training_log.csv"),
    append=True
)

In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=2e-5
)

loss = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

In [ ]:
model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[
        epoch_checkpoint,
        best_checkpoint,
        early_stopping,
        csv_logger
    ]
)

Epoch 1/10
65615/65615 [==============================] - ETA: 0s - loss: 0.3281 - accuracy: 0.8560
Epoch 1: saving model to /content/drive/MyDrive/New_BERT_Sentiment_Final/checkpoint_epoch_01

Epoch 1: val_loss improved from inf to 0.30636, saving model to /content/drive/MyDrive/New_BERT_Sentiment_Final/best_model
65615/65615 [==============================] - 4437s 67ms/step - loss: 0.3281 - accuracy: 0.8560 - val_loss: 0.3064 - val_accuracy: 0.8666
Epoch 2/10
65615/65615 [==============================] - ETA: 0s - loss: 0.2513 - accuracy: 0.8933
Epoch 2: saving model to /content/drive/MyDrive/New_BERT_Sentiment_Final/checkpoint_epoch_02

Epoch 2: val_loss improved from 0.30636 to 0.30210, saving model to /content/drive/MyDrive/New_BERT_Sentiment_Final/best_model
65615/65615 [==============================] - 4374s 67ms/step - loss: 0.2513 - accuracy: 0.8933 - val_loss: 0.3021 - val_accuracy: 0.8756
Epoch 3/10
65615/65615 [==============================] - ETA: 0s - loss: 0.1919 - a

In [ ]:
preds = model.predict(test_dataset)

y_pred = np.argmax(
    preds.logits,
    axis=1
)

18227/18227 [==============================] - 427s 23ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_sentiment,
        y_pred,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

              precision    recall  f1-score   support

    Negative       0.86      0.86      0.86    115785
     Neutral       0.89      0.89      0.89    171094
    Positive       0.61      0.68      0.64      4740

    accuracy                           0.87    291619
   macro avg       0.79      0.81      0.80    291619
weighted avg       0.88      0.87      0.88    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_sentiment, y_pred)

print(cm)

[[ 99082  16661     42]
 [ 16185 152867   2042]
 [    40   1497   3203]]


In [ ]:
tokenizer.save_pretrained(f"{SAVE_DIR_1}/best_model")

('/content/drive/MyDrive/New_BERT_Sentiment_Final/best_model/tokenizer_config.json',
 '/content/drive/MyDrive/New_BERT_Sentiment_Final/best_model/special_tokens_map.json',
 '/content/drive/MyDrive/New_BERT_Sentiment_Final/best_model/vocab.txt',
 '/content/drive/MyDrive/New_BERT_Sentiment_Final/best_model/added_tokens.json',
 '/content/drive/MyDrive/New_BERT_Sentiment_Final/best_model/tokenizer.json')

In [ ]:
import os

print(os.listdir(SAVE_DIR_1))
print(os.listdir(f"{SAVE_DIR_1}/best_model"))

['checkpoint_epoch_01', 'best_model', 'training_log.csv', 'checkpoint_epoch_02', 'checkpoint_epoch_03', 'checkpoint_epoch_04']
['variables', 'assets', 'saved_model.pb', 'fingerprint.pb', 'keras_metadata.pb', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json']
